# Lesson 13 — Execution Plans & Query Tuning

## What you'll learn
- Getting estimated vs actual execution plans in SSMS (Ctrl+L / Ctrl+M)
- Reading plan operators: Seek, Scan, Key Lookup, Nested Loops, Hash Match, Merge Join, Sort
- `SET STATISTICS IO` and `SET STATISTICS TIME` output
- SARGability — what it means and how to fix non-SARGable predicates
- Parameter sniffing — what it is and three ways to address it
- Common rewrites: correlated subquery → JOIN, function-on-column → range predicate

## Setup
Run `setup.sql` once. It creates the `lesson13` schema with an indexed copy of `SalesOrderHeader` and pre-built indexes for the examples.

**Tooling note:** run the SQL cells in this notebook with Azure Data Studio connected to the mssql-learn container (attach the SQL kernel to your saved localhost connection). For the graphical execution-plan steps, switch to SSMS and enable the actual plan (Ctrl+M) — its plan viewer shows operator costs, tooltips, and memory grant warnings that ADS does not expose as richly. SET STATISTICS IO/TIME output appears inline below each cell in ADS.

In [ ]:
USE AdventureWorks2022;
GO
SET STATISTICS IO   ON;
SET STATISTICS TIME ON;

## Concepts

### Getting plans

| Method | What you get |
|---|---|
| Ctrl+L (SSMS) | Estimated plan — no execution |
| Ctrl+M then run | Actual plan — with real row counts |
| `SET STATISTICS IO ON` | Logical reads per table (Messages tab) |
| `SET STATISTICS TIME ON` | CPU and elapsed time |

### Reading plans — key rules

1. **Read right to left, top to bottom** — data flows from leaves (scans/seeks) to root (SELECT).
2. **Arrow width** = estimated row count. Thick arrow into a cheap operator = row count mismatch.
3. **Cost %** is relative to the query, not to other queries.
4. **Yellow triangle** = optimizer warning — hover to see it.

### Operator quick reference

| Operator | Good | Investigate when |
|---|---|---|
| Index Seek | Yes — finds rows via B-tree | Rarely bad |
| Key Lookup | Sometimes | Appears per row — add INCLUDE columns |
| Clustered Index Scan | OK for full reads | Bad when a seek was expected |
| Nested Loops | Small outer set | Large outer set × large inner = slow |
| Hash Match | Large unsorted inputs | Memory spill → TempDB |
| Sort | Unavoidable sometimes | Missing index that delivers sorted rows |

## Worked examples

### Example 1 — Index Seek

SalesOrderID is the clustered key. Expect a Clustered Index Seek with very few logical reads.

In [ ]:
SELECT SalesOrderID, OrderDate, TotalDue
FROM lesson13.SalesOrderHeader
WHERE SalesOrderID = 43659;

### Example 2 — Index Scan

No useful predicate, so every row is read. Expect a Clustered Index Scan that reads every page — compare the logical reads with Example 1.

In [ ]:
SELECT COUNT(*) FROM lesson13.SalesOrderHeader;

### Example 3 — Nonclustered seek, no key lookup

The filter on CustomerID uses IX_lesson13_SOH_CustomerID; its INCLUDE columns cover OrderDate and TotalDue, so there is no Key Lookup in the plan.

In [ ]:
SELECT OrderDate, TotalDue
FROM lesson13.SalesOrderHeader
WHERE CustomerID = 11000;

### SARGability

A predicate is SARGable if the optimizer can use an index seek to satisfy it.

```sql
-- Non-SARGable (scan)                        -- SARGable rewrite (seek)
WHERE YEAR(col) = 2024                  →  WHERE col >= '2024-01-01' AND col < '2025-01-01'
WHERE CONVERT(VARCHAR,col) = '42'       →  Fix the type mismatch at the source
WHERE col LIKE '%suffix'                →  No rewrite — leading wildcard always scans
WHERE col + 1 = 5                       →  WHERE col = 4
WHERE ISNULL(col,0) = 0                 →  WHERE col = 0 OR col IS NULL
```

### Example 4 — SARGable vs non-SARGable

Run the next two cells and compare the plan operator (Scan vs Seek) and logical reads. The function on the column in the first query defeats the index on OrderDate.

In [ ]:
SELECT SalesOrderID, TotalDue
FROM lesson13.SalesOrderHeader
WHERE YEAR(OrderDate) = 2014;

In [ ]:
SELECT SalesOrderID, TotalDue
FROM lesson13.SalesOrderHeader
WHERE OrderDate >= '2014-01-01' AND OrderDate < '2015-01-01';

### `SET STATISTICS IO` output

```
Table 'SalesOrderHeader'. Scan count 1, logical reads 4, ...
```
- **logical reads** — pages read from buffer cache. Lower = better. Primary I/O metric.
- **scan count** — > 1 on the inner side of a nested-loops join.

### Example 5 — Hash Match join

Joining two large row sets — expect a Hash Match (or Merge Join) rather than Nested Loops.

In [ ]:
SELECT soh.SalesOrderID, soh.TotalDue, sod.ProductID, sod.OrderQty
FROM lesson13.SalesOrderHeader AS soh
JOIN Sales.SalesOrderDetail    AS sod ON sod.SalesOrderID = soh.SalesOrderID
WHERE soh.TerritoryID = 1;

### Parameter sniffing

SQL Server compiles the plan for the first parameter value it sees. If that value is unrepresentative:

```sql
-- Option 1: OPTIMIZE FOR UNKNOWN — use average statistics
OPTION (OPTIMIZE FOR (@param UNKNOWN))

-- Option 2: OPTIMIZE FOR specific value — if one value is typical
OPTION (OPTIMIZE FOR (@param = 1))

-- Option 3: RECOMPILE — new plan every execution (most expensive)
OPTION (RECOMPILE)
-- or: WITH RECOMPILE on the procedure
```

### Example 6 — parameter sniffing demo

The next cell creates the procedure; the cell after primes the plan cache with a small territory, then reuses that sniffed plan for a large one.

In [ ]:
CREATE OR ALTER PROCEDURE lesson13.usp_OrdersByTerritory @TerritoryID INT
AS
    SELECT SalesOrderID, OrderDate, TotalDue
    FROM lesson13.SalesOrderHeader
    WHERE TerritoryID = @TerritoryID;

In [ ]:
EXEC lesson13.usp_OrdersByTerritory @TerritoryID = 9;   -- small territory
EXEC lesson13.usp_OrdersByTerritory @TerritoryID = 1;   -- large territory (reuses sniffed plan)
-- May see a poor plan (e.g. nested loops) for territory 1 that was optimised for territory 9

To see territory 1 get its own (better) plan, flush the plan cache and re-run — demo only, never in production:

```sql
DBCC FREEPROCCACHE;
EXEC lesson13.usp_OrdersByTerritory @TerritoryID = 1;
```

### Example 7 — OPTIMIZE FOR UNKNOWN

Breaks sniffing without paying RECOMPILE on every execution: the optimizer uses average statistics instead of the first sniffed value.

In [ ]:
CREATE OR ALTER PROCEDURE lesson13.usp_OrdersByTerritory_Fixed @TerritoryID INT
AS
    SELECT SalesOrderID, OrderDate, TotalDue
    FROM lesson13.SalesOrderHeader
    WHERE TerritoryID = @TerritoryID
    OPTION (OPTIMIZE FOR (@TerritoryID UNKNOWN));

In [ ]:
EXEC lesson13.usp_OrdersByTerritory_Fixed @TerritoryID = 1;

In [ ]:
SET STATISTICS IO   OFF;
SET STATISTICS TIME OFF;

## Pitfalls
- Trusting estimated plan cost % across different queries — it's not comparable.
- Adding indexes based on missing-index hints without checking overlap with existing indexes.
- `NOLOCK` (`READ UNCOMMITTED`) hint — reads dirty data and can even skip or duplicate rows due to page splits. Almost never the right fix.
- Comparing logical reads on a cold cache — use a warm cache (run the query twice, measure the second).
- Fixing parameter sniffing with `RECOMPILE` on every execution — expensive; use `OPTIMIZE FOR UNKNOWN` first.

## Cheatsheet link
See cheatsheets/06-execution-plans.md

## Exercises
Open exercises.ipynb and work through it (use SSMS with Ctrl+M when a prompt asks about the plan). Solutions with explanations are in exercises-solutions.ipynb.